## 1.

In [2]:
import pandas as pd 
path = "data/proshares_analysis_data.xlsx"
hedge_fund_series = pd.read_excel(path, sheet_name="hedge_fund_series", index_col=0)

In [3]:
import numpy as np
ann_mean = hedge_fund_series.mean() * 12
ann_vol = hedge_fund_series.std() * np.sqrt(12)
sharpe = ann_mean / ann_vol
summary_stats = pd.DataFrame({"annualized_mean": ann_mean, "annualized volatility": ann_vol, "sharpe ratio": sharpe})
print(summary_stats)

                annualized_mean  annualized volatility  sharpe ratio
HFRIFWI Index          0.051279               0.058796      0.872151
MLEIFCTR Index         0.038535               0.055240      0.697589
MLEIFCTX Index         0.036526               0.055102      0.662880
HDG US Equity          0.026880               0.057388      0.468396
QAI US Equity          0.028811               0.049823      0.578275


## 2.

In [26]:
skewness = hedge_fund_series.skew()
kurtosis = hedge_fund_series.kurtosis()
var_fifth = hedge_fund_series.quantile(0.05)
cvar_fifth = hedge_fund_series[hedge_fund_series <= var_fifth].mean()
#max drawdown = largest peak to trough 
cum_rets = (1 + hedge_fund_series).cumprod()
rolling_max = cum_rets.cummax()
drawdown = (cum_rets - rolling_max) / rolling_max
mdd = drawdown.min()

min_dates = {}
max_dates = {}
recovery_dates = {}

for col in hedge_fund_series.columns:
    min_date = drawdown[col].idxmin()
    max_date = cum_rets.loc[:min_date, col].idxmax()
    post_min = cum_rets.loc[min_date:, col]
    recovery_date = post_min[post_min >= cum_rets.loc[max_date, col]]
    min_dates[col] = min_date
    max_dates[col] = max_date
    recovery_dates[col] = recovery_date.index[0].date()

tail_summary = pd.DataFrame({
    "skewness" : skewness, "kurtosis": kurtosis, 
    "VaR (0.05)": var_fifth, "cVaR": cvar_fifth,
    "max drawdown": mdd, "date at trough": min_dates, "date at peak": max_dates,
    "date at recovery": recovery_dates
})
print(tail_summary)

                skewness  kurtosis  VaR (0.05)      cVaR  max drawdown  \
HFRIFWI Index  -0.948272  5.657351   -0.024008 -0.035992     -0.115473   
MLEIFCTR Index -0.289982  1.630917   -0.027002 -0.034993     -0.124302   
MLEIFCTX Index -0.273477  1.589790   -0.026970 -0.034937     -0.124388   
HDG US Equity  -0.274904  1.776481   -0.029919 -0.036831     -0.140720   
QAI US Equity  -0.433508  1.449184   -0.017171 -0.030995     -0.137716   

               date at trough date at peak date at recovery  
HFRIFWI Index      2020-03-31   2019-12-31       2020-08-31  
MLEIFCTR Index     2022-09-30   2021-06-30       2024-02-29  
MLEIFCTX Index     2022-09-30   2021-06-30       2024-02-29  
HDG US Equity      2022-09-30   2021-06-30       2024-07-31  
QAI US Equity      2022-09-30   2021-06-30       2024-02-29  
